# Citadel JWT Authentication & App Role Authorization - Testing Center

## Validate JWT-based Authentication and Role-Based Access Control Across All 3 LLM API Endpoints

Use this Jupyter notebook to verify unified JWT authentication and app role authorization in Citadel Access Contracts, including:
- **Unified Security Handler**: Single `security-handler` fragment across Azure OpenAI, Universal LLM, and Unified AI APIs
- **API Key + JWT (enforced)**: Access contracts that require both API key and valid JWT Bearer token
- **App Role Authorization**: Access contracts that require specific app roles (e.g., `Model.Invoke`, `MCP.Invoke`, `Agent.Invoke`) in the JWT `roles` claim
- **All 3 API Flavors**: Testing Azure OpenAI API, Universal LLM API, and Unified AI API endpoints
- **Negative Testing**: Verifying that requests without required JWT or app roles are rejected
- **Dual Auth Flows**: Client credentials flow (service-to-service) and an explicit device-flow token fixture (delegated user identity, acquired outside Run All)

> **Note:** This notebook assumes a fully deployed Citadel Governance Hub with JWT authentication configured.
> The gateway validates tokens using APIM named values (`JWT-OpenIdConfigUrl`, `JWT-Issuer`, `JWT-AppRegistrationId`)
> or per-product custom overrides (`jwtAudience`, `jwtIssuer`, `jwtOpenIdConfigUrl`) set in the product policy.
>
> **Setup references:**
> - [JWT Authentication Guide](../guides/entraid-auth-validation.md) - Gateway-side JWT configuration (named values & custom overrides)
> - [JWT Client Identity & Permissions Guide](../guides/jwt-client-identity-permissions.md) - Direct application role assignments and user-group assignments
> - [Entra ID Setup](../bicep/infra/entra-id-setup/) - Auto-provisioning Entra ID app registration with app roles
>
> **Entra app-role assignments:** Assign roles directly to application and managed-identity service principals. Security-group assignments supply roles to signed-in users, not service-principal members.
> App roles and exposed delegated scopes share a default limit of **700 permission definitions per application or service principal**, and also count toward the **1,200-entry application manifest limit**. These are definition limits, not assignment counts. See [Microsoft Entra app roles and limits](https://learn.microsoft.com/en-us/entra/identity-platform/howto-add-app-roles-in-apps#app-role-limits).

### Shared workflow and safety

Run from the repository root or `validation`. Explicit `overrides` win over the
selected azd environment. Annotate deployed model operations with `capabilities`
(or `VALIDATION_MODEL_CAPABILITIES`); unknown capabilities block required coverage.
Image operations use `image-generation` and `image-edit`; affinity uses `responses`.
Definitions and policy XML remain visible. Preview the generated **`.bicepparam`**
in one cell, optionally edit it, then deploy it in the following cell; the shared
deploy helper compiles the actual parameter file before submission.
Generated artifacts live under ignored `validation/.runs`. Tokens/keys stay in
memory. `run.finish()` raises for required FAIL/BLOCKED/empty runs. Cleanup is off
and can only delete resources absent before this run. Cloud execution has real cost
and side effects; local source validation is not a cloud feature verdict.

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group`, `location`, the Entra `client_id` / `tenant_id` / `audience`, and the optional Key Vault / Foundry values from your active `azd` environment. Works when the accelerator was deployed with `azd up`.
- `False` — set explicit resource coordinates in `overrides`.

> The Entra **client secret** is never written to azd outputs for security reasons. Provide `entra_client_secret` manually below (or load it from your secret store).

### 0️⃣ Initialize Notebook Variables

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.

Provide `VALIDATION_ENTRA_CLIENT_SECRET` for client credentials and `VALIDATION_JWT_NO_ROLE_TOKEN` containing an actual unexpired no-role JWT. Device flow is disabled by default; device selection or its explicit example opt-in requires `VALIDATION_JWT_DEVICE_TOKEN` up front, so Run All never prompts for sign-in.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Additional scenario inputs

- `entra_client_secret` reads `VALIDATION_ENTRA_CLIENT_SECRET`. Supply it before Run All, for example `os.environ['VALIDATION_ENTRA_CLIENT_SECRET'] = secret_store_value`; use an approved secret store and never save secret values in notebook source.
- `no_role_token_fixture` reads `VALIDATION_JWT_NO_ROLE_TOKEN`. Supply it before Run All, for example `os.environ['VALIDATION_JWT_NO_ROLE_TOKEN'] = secret_store_value`; use an approved secret store and never save secret values in notebook source.
- `device_token_fixture` reads `VALIDATION_JWT_DEVICE_TOKEN`. Supply it before Run All, for example `os.environ['VALIDATION_JWT_DEVICE_TOKEN'] = secret_store_value`; use an approved secret store and never save secret values in notebook source.

### Scenario-specific input examples


Entra tenant, client and audience use azd outputs when available. Secrets and
token fixtures use the environment-variable inputs listed above.
`token_source = "client_credentials"` uses the configured confidential client;
`"device_flow"` uses an already-acquired token fixture, never an interactive
prompt during Run All. A genuine valid token without the required role is needed
for the 403 control; an invalid or expired token cannot substitute for it.


In [ ]:
from pathlib import Path
import sys
repo_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(repo_root / "shared"))
import utils
with utils.notebook_step('0️⃣ Initialize Notebook Variables', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    import os, sys, json, time, uuid, base64, requests
    from pathlib import Path
    from apimtools import APIMClientTool
    from validationtools import ValidationContext, ValidationRun, RequestTransport, ValidationBlocked
    from contracttools import write_parameters, deploy_contract, access_policy, preview
    from scenariotools import access_parameters, contract_identity, cleanup_products, policy_xml, completion_text, save_manifest

    init_from_azd = True
    environment = None  # Explicit azd environment, or the active environment.
    overrides = {}  # Explicit values win: resource_group/location/subscription_id/apim_name.
    # Supply capability metadata, not name patterns, e.g.
    # overrides["capabilities"] = {"your-deployed-model": ["chat", "responses"]}
    run_id = uuid.uuid4().hex[:12]
    cleanup_delete_contract = False

    model_override = None
    openai_api_version = "2024-12-01-preview"
    inference_api_version = "2024-05-01-preview"
    entra_client_secret = os.environ.get("VALIDATION_ENTRA_CLIENT_SECRET", "")
    no_role_token_fixture = os.environ.get("VALIDATION_JWT_NO_ROLE_TOKEN", "")
    device_token_fixture = os.environ.get("VALIDATION_JWT_DEVICE_TOKEN", "")
    use_device_flow = False  # Device-flow fixture example is opt-in; Run All never prompts.
    token_source = "client_credentials"
    use_keyvault_integration = False
    keyvault_subscription_id = ""
    keyvault_resource_group = ""
    keyvault_name = ""

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

### 1️⃣ Verify Azure CLI and Connected Subscription

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('1️⃣ Verify Azure CLI and Connected Subscription', inputs={'use_device_flow': globals().get('use_device_flow'), 'run_id': globals().get('run_id'), 'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'init_from_azd': globals().get('init_from_azd'), 'model_override': globals().get('model_override')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    run = ValidationRun("jwt-authentication")
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=init_from_azd, operation_overrides=operation_overrides, skipped_models=skipped_models)
    entra_client_secret = (entra_client_secret or context.config.get("VALIDATION_ENTRA_CLIENT_SECRET")
                           or context.config.get("ENTRA_CLIENT_SECRET", ""))
    no_role_token_fixture = no_role_token_fixture or context.config.get("VALIDATION_JWT_NO_ROLE_TOKEN", "")
    device_token_fixture = device_token_fixture or context.config.get("VALIDATION_JWT_DEVICE_TOKEN", "")
    mandatory_inputs = {
        "gateway.subscription_id": context.config.get("subscription_id"),
        "gateway.resource_group": context.config.get("resource_group"),
        "gateway.location": context.config.get("location"),
        "gateway.apim_name": context.config.get("apim_name"),
    }
    entra_tenant_id = context.config.get("entra_tenant_id") or context.account.get("tenantId")
    entra_client_id = context.config.get("entra_client_id", "")
    entra_audience = context.config.get("entra_audience") or (f"api://{entra_client_id}" if entra_client_id else "")
    mandatory_inputs.update({
        "entra.tenant_id": entra_tenant_id,
        "entra.client_id": entra_client_id,
        "entra.audience": entra_audience,
        "jwt.no_role_token_fixture": no_role_token_fixture,
    })
    if token_source == "client_credentials":
        mandatory_inputs["entra.client_secret"] = entra_client_secret
    elif token_source == "device_flow":
        mandatory_inputs["jwt.device_token_fixture"] = device_token_fixture
    else:
        raise ValueError("token_source must be client_credentials or device_flow.")
    if use_device_flow:
        mandatory_inputs["jwt.device_token_fixture"] = device_token_fixture
    if use_keyvault_integration:
        keyvault_subscription_id = keyvault_subscription_id or context.config.get("subscription_id")
        keyvault_resource_group = keyvault_resource_group or context.config.get("resource_group")
        keyvault_name = keyvault_name or context.config.get("key_vault_name")
        mandatory_inputs.update({
            "keyvault.subscription_id": keyvault_subscription_id,
            "keyvault.resource_group": keyvault_resource_group,
            "keyvault.name": keyvault_name,
        })
    utils.require_inputs(mandatory_inputs)
    utils.require_capabilities(context.catalog, context.catalog.models_for("chat"))

    try:
        parts = no_role_token_fixture.split(".")
        if len(parts) != 3:
            raise ValueError
        no_role_claims = json.loads(base64.urlsafe_b64decode(parts[1] + "=" * (-len(parts[1]) % 4)))
        if no_role_claims.get("roles") or no_role_claims.get("exp", 0) <= time.time():
            raise ValueError
    except (ValueError, TypeError, KeyError):
        raise ValueError("jwt.no_role_token_fixture must be an actual unexpired no-role JWT.") from None
    governance_hub_resource_group = context.config["resource_group"]
    location = context.config["location"]
    subscription_id = context.config["subscription_id"]
    llm_backends_config = context.config["llm_backends"]
    run_dir = repo_root / "validation" / ".runs" / "jwt-authentication" / run_id
    transport = RequestTransport(timeout=120)
    try:
        model_name = context.catalog.select('chat', override=model_override)
    except ValidationBlocked as exc:
        run.blocked("model-preflight", str(exc))
        raise

    token_endpoint = f"https://login.microsoftonline.com/{entra_tenant_id}/oauth2/v2.0/token"
    keyvault_subscription_id = keyvault_subscription_id or subscription_id
    keyvault_resource_group = keyvault_resource_group or governance_hub_resource_group
    keyvault_name = keyvault_name or context.config.get("key_vault_name", "")
    if use_keyvault_integration and not keyvault_name:
        raise ValueError("Enabled Key Vault integration requires keyvault_name.")

    utils.format_output('1️⃣ Verify Azure CLI and Connected Subscription', {'value_1': {"environment": environment or "active", "resource_group": governance_hub_resource_group,
           "location": location, "subscription_id": subscription_id}}, status='info', part='RESULT')
    utils.format_output('1️⃣ Verify Azure CLI and Connected Subscription', {'value_1': context.catalog.report()}, status='info', part='RESULT')

    run_dir.mkdir(parents=True, exist_ok=True)


<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected with JWT named values configured.

### 2️⃣ Initialize APIM Client Tool

Prepare the 2️⃣ Initialize APIM Client Tool state shown below for its subsequent feature-specific requests and assertions.

**Input:** `governance_hub_resource_group`, `model_name`, `openai_api_version`, `inference_api_version`

**Output:** `apimClientTool`, `openai_api`, `universal_api`, `unified_api`, `gateway_url`, `azure_endpoint` are available to the following cells.


In [ ]:
with utils.notebook_step('2️⃣ Initialize APIM Client Tool', inputs={'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'model_name': globals().get('model_name'), 'openai_api_version': globals().get('openai_api_version'), 'inference_api_version': globals().get('inference_api_version')}, expected='`apimClientTool`, `openai_api`, `universal_api`, `unified_api`, `gateway_url`, `azure_endpoint` are available to the following cells.', namespace=globals()):
    apimClientTool = APIMClientTool(governance_hub_resource_group, apim_resource_name=context.config["apim_name"])
    apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)
    openai_api = apimClientTool.discover_api_exact("openai")
    universal_api = apimClientTool.discover_api_exact("models")
    unified_api = apimClientTool.discover_api_exact("unified-ai")
    gateway_url = str(apimClientTool.apim_resource_gateway_url).rstrip("/")
    azure_endpoint = gateway_url + "/"
    apim_resource_gateway_url = gateway_url
    apim_name = apimClientTool.apim_resource_name
    preexisting_products = {p.name for p in apimClientTool.client.product.list_by_service(
        governance_hub_resource_group, apim_name)}
    owned_products = set()

    from urllib.parse import quote
    encoded_model = quote(model_name, safe="")
    openai_chat_url = f"{openai_api['url'].rstrip('/')}/deployments/{encoded_model}/chat/completions?api-version={openai_api_version}"
    universal_llm_chat_url = f"{universal_api['url'].rstrip('/')}/chat/completions?api-version={inference_api_version}"
    unified_ai_openai_url = f"{unified_api['url'].rstrip('/')}/openai/deployments/{encoded_model}/chat/completions?api-version={openai_api_version}"
    unified_ai_models_url = f"{unified_api['url'].rstrip('/')}/models/chat/completions?api-version={inference_api_version}"


<a id='3'></a>
### 3️⃣ Acquire JWT Token

Acquire a JWT token from the identity provider using client credentials flow.
For role-bearing application tokens, assign `Model.Invoke` directly to the service principal for `entra_client_id`. Managed identities likewise require direct assignments to their own service principals; group membership does not supply application roles.
Entra ID can issue a token with the correct audience but without `roles`. Such a token can exercise JWT-only authentication, but the role-enforced success test requires `Model.Invoke`.

### 3️⃣ Acquire JWT Token

Inspect or exercise 3️⃣ Acquire JWT Token using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `entra_client_id`, `entra_audience`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('3️⃣ Acquire JWT Token', inputs={'entra_client_id': globals().get('entra_client_id'), 'entra_audience': globals().get('entra_audience')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    jwt_token = None
    if token_source == "client_credentials":

        token_data = {
            "grant_type": "client_credentials",
            "client_id": entra_client_id,
            "client_secret": entra_client_secret,
            "scope": f"{entra_audience}/.default"
        }

        utils.print_info(f"Requesting JWT token from: {token_endpoint}")
        utils.print_info(f"Client ID: {entra_client_id}")
        utils.print_info(f"Audience: {entra_audience}")

        try:
            token_response = utils.http_call(requests.post, token_endpoint, data=token_data, timeout=30)

            if token_response.status_code == 200:
                token_json = token_response.json()
                jwt_token = token_json["access_token"]
                expires_in = token_json.get("expires_in", "unknown")

                utils.print_ok(f"JWT token acquired successfully!")
                utils.print_info(f"Token length: {len(jwt_token)} characters")
                utils.print_info(f"Expires in: {expires_in} seconds")

                # Decode token payload to inspect claims
                token_parts = jwt_token.split('.')
                if len(token_parts) >= 2:
                    payload = token_parts[1] + '=' * (4 - len(token_parts[1]) % 4)
                    decoded = json.loads(base64.b64decode(payload))
                    utils.print_info(f"Token issuer: {decoded.get('iss', 'N/A')}")
                    utils.print_info(f"Token audience: {decoded.get('aud', 'N/A')}")
                    utils.print_info(f"Token app ID (azp): {decoded.get('azp', 'N/A')}")

                    # Display app roles from the token
                    token_roles = decoded.get('roles', [])
                    if token_roles:
                        utils.print_ok(f"Token roles: {', '.join(token_roles)}")
                    else:
                        utils.print_info("Token roles: none. Assign gateway app roles directly to the client service principal.")
                        utils.print_info("Service principals, including managed identities, do not inherit app roles from group membership.")
                        utils.print_info(f"  To assign a role: az rest --method POST --uri 'https://graph.microsoft.com/v1.0/servicePrincipals/{{client-sp-id}}/appRoleAssignments' ...")
            else:
                utils.print_error(f"Failed to acquire token: {token_response.status_code}")
                utils.print_error(f"Response: {token_response.text[:500]}")
        except Exception as e:
            utils.print_error(f"Error acquiring JWT token: {e}")

        utils.require_inputs({"jwt.acquired_client_credentials_token": jwt_token})


<a id='3b'></a>
### 3️⃣.b Acquire JWT Token via Device Code Flow (Interactive)

For **delegated user identity** scenarios, acquire the `access_as_user` token separately and provide it through `VALIDATION_JWT_DEVICE_TOKEN`. Run All requires this fixture up front and does not initiate sign-in. The original MSAL acquisition example below remains visible for reference but is unreachable after the fixture preflight.
App roles assigned directly to the **user** or to a **security group containing the user** can appear in the delegated token's `roles` claim. This group assignment path does not apply to service principals or managed identities.

> **Prerequisites:**
> - MSAL is included in [Validation Environment Setup](citadel-environment-setup.ipynb); select the same `.venv-validation-local` kernel.
> - The app registration must have `isFallbackPublicClient` enabled (or use the Entra ID portal to enable "Allow public client flows")
> - The user must be assigned an app role (e.g., `Model.Invoke`) via Entra ID portal or CLI

### 3️⃣.b Acquire JWT Token via Device Code Flow (Interactive)

Inspect or exercise 3️⃣.b Acquire JWT Token via Device Code Flow (Interactive) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `use_device_flow`, `entra_client_id`, `entra_tenant_id`, `parts`, `entra_audience`, `payload`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('3️⃣.b Acquire JWT Token via Device Code Flow (Interactive)', inputs={'use_device_flow': globals().get('use_device_flow'), 'entra_client_id': globals().get('entra_client_id'), 'entra_tenant_id': globals().get('entra_tenant_id'), 'parts': globals().get('parts'), 'entra_audience': globals().get('entra_audience'), 'payload': globals().get('payload')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    # Device code flow using MSAL (interactive user sign-in)
    # Supply the delegated fixture before opting into the device-flow example.
    if use_device_flow or token_source == "device_flow":
        utils.require_inputs({"jwt.device_token_fixture": device_token_fixture})
    device_flow_token = device_token_fixture or None

    if use_device_flow and not device_flow_token:
        # Device code flow requires the app to allow public client flows.
        # Enable it if not already set (idempotent PATCH).
        # utils.print_info("Ensuring app registration allows public client flows...")
        # utils.run(
        #     f"az ad app update --id {entra_client_id} --is-fallback-public-client true",
        #     "Public client flows enabled",
        #     "Failed to enable public client flows (you may need to set it manually in Entra ID portal)"
        # )

        try:
            import msal

            app = msal.PublicClientApplication(
                entra_client_id,
                authority=f"https://login.microsoftonline.com/{entra_tenant_id}"
            )

            # Initiate device code flow with the gateway's delegated scope
            flow = app.initiate_device_flow(scopes=[f"{entra_audience}/access_as_user"])
            if "user_code" not in flow:
                utils.print_error(f"Failed to create device flow: {json.dumps(flow, indent=2)}")
            else:
                utils.format_output('3️⃣.b Acquire JWT Token via Device Code Flow (Interactive)', {'value_1': flow["message"]}, status='info', part='RESULT')  # Displays: "To sign in, use a web browser to open..."

                result = app.acquire_token_by_device_flow(flow)

                if "access_token" in result:
                    device_flow_token = result["access_token"]
                    utils.print_ok(f"Device flow token acquired!")
                    utils.print_info(f"Token length: {len(device_flow_token)} characters")

                    # Decode and display claims including roles
                    parts = device_flow_token.split('.')
                    if len(parts) >= 2:
                        payload = parts[1] + '=' * (4 - len(parts[1]) % 4)
                        decoded = json.loads(base64.b64decode(payload))
                        utils.print_info(f"User: {decoded.get('preferred_username', decoded.get('upn', 'N/A'))}")
                        utils.print_info(f"Audience: {decoded.get('aud', 'N/A')}")
                        roles = decoded.get('roles', [])
                        if roles:
                            utils.print_ok(f"Roles: {', '.join(roles)}")
                        else:
                            utils.print_info(f"Roles: none (assign roles to the user via Entra ID)")
                else:
                    utils.print_error(f"Token acquisition failed: {result.get('error_description', result.get('error', 'unknown'))}")
        except ImportError:
            utils.print_error("MSAL not installed. Run citadel-environment-setup.ipynb, restart, and select the same .venv-validation-local kernel.")
        except Exception as e:
            utils.print_error(f"Device flow error: {e}")
    else:
        utils.print_info("Using the supplied device-flow fixture; no sign-in prompt." if device_flow_token else
                         "Device flow is disabled. Provide VALIDATION_JWT_DEVICE_TOKEN and explicitly select device_flow to enable.")


<a id='3c'></a>
### 3️⃣.c Inspect Issued JWT Tokens

Decode and display the full JWT payload for any tokens acquired above. Use [jwt.ms](https://jwt.ms) or [jwt.io](https://jwt.io) for interactive inspection.

### 3️⃣.c Inspect Issued JWT Tokens

Inspect or exercise 3️⃣.c Inspect Issued JWT Tokens using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `roles`, `parts`, `payload`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('3️⃣.c Inspect Issued JWT Tokens', inputs={'roles': globals().get('roles'), 'parts': globals().get('parts'), 'payload': globals().get('payload')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    def inspect_jwt(token, label="JWT Token"):
        """Decode and pretty-print a JWT token's header and payload."""
        if not token:
            utils.print_info(f"[{label}] No token to inspect.")
            return

        parts = token.split('.')
        if len(parts) < 2:
            utils.print_error(f"[{label}] Invalid JWT format (expected 3 parts, got {len(parts)})")
            return

        def decode_part(part):
            padded = part + '=' * (4 - len(part) % 4)
            return json.loads(base64.urlsafe_b64decode(padded))

        header = decode_part(parts[0])
        payload = decode_part(parts[1])

        utils.print_info(f"\n{'='*80}")
        utils.print_info(f"🔍 {label}")
        utils.print_info(f"{'='*80}")

        utils.print_info(f"\n--- Header ---")
        utils.format_output('3️⃣.c Inspect Issued JWT Tokens', {'value_1': json.dumps(header, indent=2)}, status='info', part='RESULT')

        utils.print_info(f"\n--- Payload ---")
        utils.format_output('3️⃣.c Inspect Issued JWT Tokens', {'value_1': json.dumps(payload, indent=2)}, status='info', part='RESULT')

        # Highlight key claims
        utils.print_info(f"\n--- Key Claims ---")
        utils.print_info(f"  aud (audience):  {payload.get('aud', 'N/A')}")
        utils.print_info(f"  iss (issuer):    {payload.get('iss', 'N/A')}")
        utils.print_info(f"  azp (client):    {payload.get('azp', 'N/A')}")
        utils.print_info(f"  sub (subject):   {payload.get('sub', 'N/A')}")
        utils.print_info(f"  upn/email:       {payload.get('preferred_username', payload.get('upn', payload.get('email', 'N/A')))}")

        roles = payload.get('roles', [])
        if roles:
            utils.print_ok(f"  roles:           {', '.join(roles)}")
        else:
            utils.print_info(f"  roles:           none")

        # Token lifetime
        import datetime
        iat = payload.get('iat')
        exp = payload.get('exp')
        if iat and exp:
            issued = datetime.datetime.fromtimestamp(iat, tz=datetime.timezone.utc)
            expires = datetime.datetime.fromtimestamp(exp, tz=datetime.timezone.utc)
            utils.print_info(f"  issued:          {issued.isoformat()}")
            utils.print_info(f"  expires:         {expires.isoformat()}")
            utils.print_info(f"  lifetime:        {(expires - issued).total_seconds() / 60:.0f} minutes")

        # Token bytes are deliberately never printed or persisted.

    # Inspect all acquired tokens
    inspect_jwt(jwt_token, "Client Credentials Token (§3)")

    if device_flow_token:
        inspect_jwt(device_flow_token, "Device Code Flow Token (§3b)")


<a id='3d'></a>
### 3️⃣.d Select Active Token for Testing

Choose which token to use for all subsequent tests. Change `token_source` to switch between:
- `"client_credentials"` — Service principal token from §3 (service-to-service)
- `"device_flow"` — Delegated user token from the required `VALIDATION_JWT_DEVICE_TOKEN` fixture in §3b (no Run All sign-in prompt)

### 3️⃣.d Select Active Token for Testing

Inspect or exercise 3️⃣.d Select Active Token for Testing using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `parts`, `roles`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('3️⃣.d Select Active Token for Testing', inputs={'parts': globals().get('parts'), 'roles': globals().get('roles')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    # ============================================================================
    # TOKEN SELECTOR — Change this value to switch which token is used in all tests
    # ============================================================================
    # token_source is configured in the first cell.

    # Resolve the active token based on selection
    if token_source == "device_flow":
        if device_flow_token:
            active_token = device_flow_token
            utils.print_ok(f"Active token: Device Code Flow")
        else:
            utils.print_error("Device flow token not available. Run §3b first or switch to 'client_credentials'.")
            active_token = None
    elif token_source == "client_credentials":
        if jwt_token:
            active_token = jwt_token
            utils.print_ok(f"Active token: Client Credentials (§3)")
        else:
            utils.print_error("Client credentials token not available. Run §3 first.")
            active_token = None
    else:
        utils.print_error(f"Unknown token_source: '{token_source}'. Use 'client_credentials' or 'device_flow'.")
        active_token = None

    utils.require_inputs({"jwt.active_token": active_token})

    # Display summary of the selected token
    if active_token:
        parts = active_token.split('.')
        if len(parts) >= 2:
            pl = parts[1] + '=' * (4 - len(parts[1]) % 4)
            claims = json.loads(base64.urlsafe_b64decode(pl))
            roles = claims.get('roles', [])
            utils.print_info(f"  Source:   {token_source}")
            utils.print_info(f"  Audience: {claims.get('aud', 'N/A')}")
            utils.print_info(f"  Identity: {claims.get('azp', claims.get('preferred_username', 'N/A'))}")
            utils.print_info(f"  Roles:    {', '.join(roles) if roles else 'none'}")


---
<a id='use-case-1'></a>
## 🛡️ Use Case 1: JWT-Enforced Access Contract

This use case creates an access contract that requires **both API key and JWT token** for all requests.
The `jwtRequired` variable is set to `"true"` in the product policy, which triggers the `security-handler`
fragment to enforce JWT validation across all 3 API endpoints.

---

<a id='4.1'></a>
### 4️⃣.1 Define JWT Access Contract Configuration

### 4️⃣.1 Define JWT Access Contract Configuration

Inspect or exercise 4️⃣.1 Define JWT Access Contract Configuration using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `run_id`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('4️⃣.1 Define JWT Access Contract Configuration', inputs={'run_id': globals().get('run_id')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    timestamp = run_id

    # JWT-Enforced Access Contract Configuration
    jwt_contract = {
        "name": f"jwt-auth-contract-{timestamp}",
        "business_unit": "Security",
        "use_case_name": "JwtAuth-" + run_id,
        "environment": "DEV",
        "use_keyvault": use_keyvault_integration,
        "endpoint_secret": "SEC-JWT-LLM-ENDPOINT",
        "apikey_secret": "SEC-JWT-LLM-KEY",
        "description": "Security JWT Auth - Enforced Valid JWT + API Key across all endpoints"
    }

    jwt_product_id = f"LLM-{jwt_contract['business_unit']}-{jwt_contract['use_case_name']}-{jwt_contract['environment']}"

    utils.print_info(f"JWT Access Contract Configuration:")
    utils.print_info(f"  Business Unit: {jwt_contract['business_unit']}")
    utils.print_info(f"  Use Case: {jwt_contract['use_case_name']}")
    utils.print_info(f"  JWT Enforcement: ENABLED")
    utils.print_info(f"  Product ID: {jwt_product_id}")


<a id='4.2'></a>
### 4️⃣.2 Create JWT Product Policy

Generate a product policy XML that enables JWT authentication via the unified `security-handler` fragment.

### 4️⃣.2 Create JWT Product Policy

Inspect or exercise 4️⃣.2 Create JWT Product Policy using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `model_name`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('4️⃣.2 Create JWT Product Policy', inputs={'model_name': globals().get('model_name')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    jwt_product_policy = access_policy([model_name], 5000, 100000, "Monthly", jwt_required=True)
    utils.format_output('4️⃣.2 Create JWT Product Policy', {'jwt_product_policy': jwt_product_policy}, status='info', part='RESULT')


<a id='4.3'></a>
### 4️⃣.3 Create Parameter File and Deploy JWT Access Contract

### 4️⃣.3 Create Parameter File and Deploy JWT Access Contract

Generate the editable parameter artifact for 4️⃣.3 Create Parameter File and Deploy JWT Access Contract and display the proposed configuration without deploying it.

**Input:** `jwt_contract`, `jwt_product_policy`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('4️⃣.3 Create Parameter File and Deploy JWT Access Contract', inputs={'jwt_contract': globals().get('jwt_contract'), 'jwt_product_policy': globals().get('jwt_product_policy'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    jwt_parameters = access_parameters(context, apimClientTool, jwt_contract, jwt_product_policy,
        key_vault={"subscriptionId": keyvault_subscription_id, "resourceGroupName": keyvault_resource_group, "name": keyvault_name} if use_keyvault_integration else None)

    jwt_params_file = write_parameters(
        repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
        jwt_parameters, run_dir / "jwt")
    utils.format_output('4️⃣.3 Create Parameter File and Deploy JWT Access Contract', {'value_1': preview(jwt_params_file)}, status='info', part='RESULT')


<a id='4.4'></a>
### 4️⃣.4 Retrieve API Key for JWT Access Contract

### 4️⃣.4 Retrieve API Key for JWT Access Contract

Deploy the reviewed artifact for 4️⃣.4 Retrieve API Key for JWT Access Contract, then inspect its real resource outputs and ownership.

**Input:** `jwt_params_file`, `location`, `preexisting_products`, `owned_products`, `jwt_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('4️⃣.4 Retrieve API Key for JWT Access Contract', inputs={'jwt_params_file': globals().get('jwt_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'jwt_contract': globals().get('jwt_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    jwt_outputs = deploy_contract(jwt_params_file, location, jwt_contract["name"],
        required_outputs=("products", "subscriptions"))
    jwt_product_id, jwt_subscription_name = contract_identity(
        jwt_outputs, preexisting_products, owned_products)
    jwt_api_key = apimClientTool.get_subscription_key(jwt_subscription_name)


---
## 🧪 Test Suite: JWT Authentication Across All 3 API Endpoints

The following tests validate the unified `security-handler` behavior:
1. **API Key + JWT** on each endpoint (should succeed)
2. **API Key only** on each endpoint (should fail with 401 - JWT required)
3. **JWT only** without API key (should fail with 401 - API key required)
4. **Invalid JWT** with valid API key (should fail with 401 - invalid token)

---

<a id='5.1'></a>
### 5️⃣.1 Define Test Endpoints and Payloads

### 5️⃣.1 Define Test Endpoints and Payloads

Exercise 5️⃣.1 Define Test Endpoints and Payloads and compare the actual response/evidence with the expected feature behavior.

**Input:** `openai_chat_url`, `universal_llm_chat_url`, `unified_ai_openai_url`, `unified_ai_models_url`, `model_name`, `claims`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('5️⃣.1 Define Test Endpoints and Payloads', inputs={'openai_chat_url': globals().get('openai_chat_url'), 'universal_llm_chat_url': globals().get('universal_llm_chat_url'), 'unified_ai_openai_url': globals().get('unified_ai_openai_url'), 'unified_ai_models_url': globals().get('unified_ai_models_url'), 'model_name': globals().get('model_name'), 'claims': globals().get('claims')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    # Define the 3 API endpoint configurations
    api_endpoints = [
        {
            "name": "Azure OpenAI API",
            "url": openai_chat_url,
            "payload": {
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                    {"role": "user", "content": "Say hello in exactly 3 words."}
                ],
                "max_tokens": 20,
                "temperature": 0.3
            }
        },
        {
            "name": "Universal LLM API",
            "url": universal_llm_chat_url,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                    {"role": "user", "content": "Say hello in exactly 3 words."}
                ],
                "max_tokens": 20,
                "temperature": 0.3
            }
        },
        {
            "name": "Unified AI API (OpenAI format)",
            "url": unified_ai_openai_url,
            "payload": {
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                    {"role": "user", "content": "Say hello in exactly 3 words."}
                ],
                "max_tokens": 20,
                "temperature": 0.3,
                "model": model_name
            }
        },
        {
            "name": "Unified AI API (Models/Inference format)",
            "url": unified_ai_models_url,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                    {"role": "user", "content": "Say hello in exactly 3 words."}
                ],
                "max_tokens": 20,
                "temperature": 0.3
            }
        }
    ]

    utils.print_info(f"Defined {len(api_endpoints)} API endpoints for testing:")
    for ep in api_endpoints:
        utils.print_info(f"  - {ep['name']}: {ep['url'][:80]}...")

    def token_claims(token):
        if not token or len(token.split(".")) != 3:
            return {}
        part = token.split(".")[1]
        try:
            claims = json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4)))
            return claims if isinstance(claims, dict) else {}
        except (ValueError, TypeError):
            return {}

    def auth_case(case, api_key_value, token, expected_status, error_code=None, required_role=None):
        for ep in api_endpoints:
            case_id = f"{case}/{ep['name']}"
            if (expected_status == 200 or case == "04-jwt-without-key") and not token:
                run.blocked(case_id, "Acquire a valid client-credentials or delegated token first.")
                continue
            if required_role and required_role not in token_claims(token).get("roles", []):
                run.blocked(case_id, f"Assign {required_role} and acquire a fresh token.")
                continue
            headers = {"Content-Type": "application/json"}
            if api_key_value:
                headers["api-key"] = api_key_value
            if token:
                headers["Authorization"] = f"Bearer {token}"
            response = transport.request("POST", ep["url"], headers=headers, json=ep["payload"])
            data = response.json_body if isinstance(response.json_body, dict) else {}
            err = data.get("error", {})
            observed_code = err.get("code", "") if isinstance(err, dict) else ""
            if expected_status == 200:
                body_ok = bool(data.get("choices", [{}])[0].get("message", {}).get("content"))
            else:
                # JWT validation errors may be APIM's JSON {statusCode,message} shape.
                body_ok = observed_code == error_code if error_code else bool(
                    observed_code or (data.get("statusCode") == expected_status and data.get("message")))
            run.check(case_id, response.status_code == expected_status and body_ok and not response.error,
                expected=f"HTTP {expected_status}; {error_code or 'valid body/policy error'}",
                actual=response.status_code, api=ep["name"], model=model_name,
                error_code=observed_code, transport_error=response.error, duration=response.duration)


<a id='5.2'></a>
### 5️⃣.2 Test 1: API Key + JWT Token (Should Succeed ✅)

Send requests to all 3 API endpoints with both API key and valid JWT Bearer token.

### 5️⃣.2 Test 1: API Key + JWT Token (Should Succeed ✅)

Inspect or exercise 5️⃣.2 Test 1: API Key + JWT Token (Should Succeed ✅) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('5️⃣.2 Test 1: API Key + JWT Token (Should Succeed ✅)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    auth_case("01-key-and-jwt", jwt_api_key, active_token, 200)


<a id='5.3'></a>
### 5️⃣.3 Test 2: API Key Only - No JWT (Should Fail ❌)

Send requests with only the API key (no JWT token). Since the access contract has `jwtRequired=true`,
these should be rejected with HTTP 401.

### 5️⃣.3 Test 2: API Key Only - No JWT (Should Fail ❌)

Inspect or exercise 5️⃣.3 Test 2: API Key Only - No JWT (Should Fail ❌) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('5️⃣.3 Test 2: API Key Only - No JWT (Should Fail ❌)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    auth_case("02-key-without-jwt", jwt_api_key, None, 401, "jwt_required")


<a id='5.4'></a>
### 5️⃣.4 Test 3: Invalid JWT Token (Should Fail ❌)

Send requests with a valid API key but an invalid/expired JWT token.
These should be rejected with HTTP 401.

### 5️⃣.4 Test 3: Invalid JWT Token (Should Fail ❌)

Inspect or exercise 5️⃣.4 Test 3: Invalid JWT Token (Should Fail ❌) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('5️⃣.4 Test 3: Invalid JWT Token (Should Fail ❌)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    auth_case("03-invalid-jwt", jwt_api_key, "INVALID-TEST-TOKEN-FOR-VALIDATION", 401)


<a id='5.5'></a>
### 5️⃣.5 Test 4: JWT Only - No API Key (Should Fail ❌)

Send requests with only a JWT token (no API key). APIM requires a valid subscription key,
so these should be rejected.

### 5️⃣.5 Test 4: JWT Only - No API Key (Should Fail ❌)

Inspect or exercise 5️⃣.5 Test 4: JWT Only - No API Key (Should Fail ❌) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('5️⃣.5 Test 4: JWT Only - No API Key (Should Fail ❌)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    auth_case("04-jwt-without-key", None, active_token, 401)


---
<a id='use-case-2'></a>
## 🛡️ Use Case 2: Role-Enforced Access Contract

> 🔄 [Back to Select Active Token](#3d) — switch between device flow and client credentials before running tests.

This use case creates an access contract that requires **both JWT token and a specific app role**.
The `requiredRoles` variable is set in the product policy, which triggers the `security-handler`
fragment to verify the token's `roles` claim contains at least one of the required roles.

**Prerequisite:** For client credentials, assign `Model.Invoke` directly to the service principal used in §3 through application-permission admin consent or a direct Graph/CLI app-role assignment. Managed identities also require direct assignments; group membership does not supply their roles.
For device flow, assign `Model.Invoke` to the signed-in user directly or through a users' security group, and provide a fresh delegated token.

---

<a id='6.1'></a>
### 6️⃣.1 Define Role-Enforced Access Contract Configuration

### 6️⃣.1 Define Role-Enforced Access Contract Configuration

Prepare the 6️⃣.1 Define Role-Enforced Access Contract Configuration state shown below for its subsequent feature-specific requests and assertions.

**Input:** `run_id`, `timestamp`

**Output:** `role_contract`, `role_product_id` are available to the following cells.


In [ ]:
with utils.notebook_step('6️⃣.1 Define Role-Enforced Access Contract Configuration', inputs={'run_id': globals().get('run_id'), 'timestamp': globals().get('timestamp')}, expected='`role_contract`, `role_product_id` are available to the following cells.', namespace=globals()):
    # Role-Enforced Access Contract Configuration
    role_contract = {
        "name": f"role-auth-contract-{timestamp}",
        "business_unit": "Security",
        "use_case_name": "RoleAuth-" + run_id,
        "environment": "DEV",
        "use_keyvault": use_keyvault_integration,
        "endpoint_secret": "SEC-ROLE-LLM-ENDPOINT",
        "apikey_secret": "SEC-ROLE-LLM-KEY",
        "description": "Security Role Auth - JWT + App Role enforcement across all endpoints"
    }

    role_product_id = f"LLM-{role_contract['business_unit']}-{role_contract['use_case_name']}-{role_contract['environment']}"

    utils.print_info(f"Role Access Contract Configuration:")
    utils.print_info(f"  Business Unit: {role_contract['business_unit']}")
    utils.print_info(f"  Use Case: {role_contract['use_case_name']}")
    utils.print_info(f"  JWT Enforcement: ENABLED")
    utils.print_info(f"  Required Role: Model.Invoke")
    utils.print_info(f"  Product ID: {role_product_id}")


<a id='6.2'></a>
### 6️⃣.2 Create Role-Enforced Product Policy and Deploy

### 6️⃣.2 Create Role-Enforced Product Policy and Deploy

Generate the editable parameter artifact for 6️⃣.2 Create Role-Enforced Product Policy and Deploy and display the proposed configuration without deploying it.

**Input:** `role_contract`, `model_name`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('6️⃣.2 Create Role-Enforced Product Policy and Deploy', inputs={'role_contract': globals().get('role_contract'), 'model_name': globals().get('model_name'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    role_product_policy = access_policy([model_name], 5000, 100000, "Monthly",
        jwt_required=True, inbound_extra='<set-variable name="requiredRoles" value="Model.Invoke" />')
    utils.format_output('6️⃣.2 Create Role-Enforced Product Policy and Deploy', {'role_product_policy': role_product_policy}, status='info', part='RESULT')

    role_parameters = access_parameters(context, apimClientTool, role_contract, role_product_policy,
        key_vault={"subscriptionId": keyvault_subscription_id, "resourceGroupName": keyvault_resource_group, "name": keyvault_name} if use_keyvault_integration else None)

    role_params_file = write_parameters(
        repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
        role_parameters, run_dir / "role")
    utils.format_output('6️⃣.2 Create Role-Enforced Product Policy and Deploy', {'value_1': preview(role_params_file)}, status='info', part='RESULT')


<a id='6.3'></a>
### 6️⃣.3 Retrieve API Key for Role-Enforced Access Contract

### 6️⃣.3 Retrieve API Key for Role-Enforced Access Contract

Deploy the reviewed artifact for 6️⃣.3 Retrieve API Key for Role-Enforced Access Contract, then inspect its real resource outputs and ownership.

**Input:** `role_params_file`, `location`, `preexisting_products`, `owned_products`, `role_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('6️⃣.3 Retrieve API Key for Role-Enforced Access Contract', inputs={'role_params_file': globals().get('role_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'role_contract': globals().get('role_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    role_outputs = deploy_contract(role_params_file, location, role_contract["name"],
        required_outputs=("products", "subscriptions"))
    role_product_id, role_subscription_name = contract_identity(
        role_outputs, preexisting_products, owned_products)
    role_api_key = apimClientTool.get_subscription_key(role_subscription_name)


<a id='7.1'></a>
### 7️⃣.1 Test 5: API Key + JWT with Correct Role (Should Succeed ✅)

Send requests with both API key and JWT token that contains the required `Model.Invoke` role.
The `security-handler` should validate the role and allow the request.

### 7️⃣.1 Test 5: API Key + JWT with Correct Role (Should Succeed ✅)

Inspect or exercise 7️⃣.1 Test 5: API Key + JWT with Correct Role (Should Succeed ✅) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('7️⃣.1 Test 5: API Key + JWT with Correct Role (Should Succeed ✅)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    auth_case("05-role-bearing-jwt", role_api_key, active_token, 200, required_role="Model.Invoke")


<a id='7.2'></a>
### 7️⃣.2 Test 6: Genuine JWT Without Required Role (403 insufficient_role)

Supply `VALIDATION_JWT_NO_ROLE_TOKEN` from a second identity without `Model.Invoke`.
It must first succeed against the JWT-only contract on each API, proving audience,
issuer and signature validation, then fail with **403 insufficient_role** against
the role contract. Missing or invalid fixtures are **BLOCKED**, never replaced by
an invalid token. The separate invalid-token case remains in §5.4.


### 7️⃣.2 Test 6: Genuine JWT Without Required Role (403 insufficientrole)

Inspect or exercise 7️⃣.2 Test 6: Genuine JWT Without Required Role (403 insufficientrole) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `api_endpoints`, `headers`, `case_id`, `ep`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('7️⃣.2 Test 6: Genuine JWT Without Required Role (403 insufficientrole)', inputs={'api_endpoints': globals().get('api_endpoints'), 'headers': globals().get('headers'), 'case_id': globals().get('case_id'), 'ep': globals().get('ep')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    # Prove validity using the JWT-only product before testing the role-enforced product.
    no_role_token = no_role_token_fixture or (
        active_token if "Model.Invoke" not in token_claims(active_token).get("roles", []) else "")
    for ep in api_endpoints:
        case_id = f"06-missing-role/{ep['name']}"
        claims = token_claims(no_role_token)
        if (not no_role_token or "Model.Invoke" in claims.get("roles", [])
            or not isinstance(claims.get("exp"), (int, float)) or claims["exp"] <= time.time()):
            run.blocked(case_id, "Set VALIDATION_JWT_NO_ROLE_TOKEN to a fresh, genuine token for the same audience from an identity without Model.Invoke. No invalid-token fallback.")
            continue
        headers = {"api-key": jwt_api_key, "Authorization": f"Bearer {no_role_token}"}
        control = transport.request("POST", ep["url"], headers=headers, json=ep["payload"])
        control_data = control.json_body if isinstance(control.json_body, dict) else {}
        if control.status_code != 200 or not control_data.get("choices", [{}])[0].get("message", {}).get("content"):
            run.blocked(case_id, "No-role token failed the non-role JWT product control; role rejection is not proven.")
            continue
        headers["api-key"] = role_api_key
        denied = transport.request("POST", ep["url"], headers=headers, json=ep["payload"])
        data = denied.json_body if isinstance(denied.json_body, dict) else {}
        run.check(case_id, denied.status_code == 403 and data.get("error", {}).get("code") == "insufficient_role",
            expected="Valid token accepted by JWT-only product, then 403 insufficient_role",
            actual=denied.status_code, control_status=control.status_code, api=ep["name"])


<a id='7.3'></a>
### 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌)

Send requests with only the API key (no JWT). Since the product has `jwtRequired=true`,
the request should be rejected with 401 before role checking even occurs.

### 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌)

Prepare the 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌) state shown below for its subsequent feature-specific requests and assertions.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The visible 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌) inspection output.


In [ ]:
with utils.notebook_step('7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌)', inputs={}, expected='The visible 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌) inspection output.', namespace=globals()):
    auth_case("07-role-product-no-jwt", role_api_key, None, 401, "jwt_required")


---
## 📊 Results Summary
---

### 📊 Results Summary

Collect the final 📊 Results Summary verdict without discarding earlier failed or blocked required cases.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('📊 Results Summary', inputs={}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    run.finish()


<a id='cleanup'></a>
### Cleanup (Optional)

Remove the test JWT access contract from APIM created during this notebook session.

> **Note:** This will not delete any Entra ID app registrations or APIM named values.

### Cleanup (Optional)

Apply the explicitly enabled Cleanup (Optional) action only to resources owned by this run.

**Input:** `owned_products`, `cleanup_delete_contract`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('Cleanup (Optional)', inputs={'owned_products': globals().get('owned_products'), 'cleanup_delete_contract': globals().get('cleanup_delete_contract')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    cleanup_products(apimClientTool, owned_products, enabled=cleanup_delete_contract)
